# Visualizing the smoothing/spread chain and onset-time fidelity

This notebook walks through the behavior of
`DynaSDBase.get_onset_and_spread` end-to-end on:

1. **Synthetic binary `sz_prob` inputs** — clean steps and realistic
   post-thresholded patterns (drop-outs, flicker, cascade). These
   match the tests in `tests/test_windowing_smoothing_spec.py` § 7.4
   one-to-one and let you see the smoothing → threshold → spread
   chain in plot form.
2. **A full simulated polyspike seizure** — runs `HFER.forward(X)`
   then `get_onset_and_spread`, overlays the planted onset on top
   of the detected onset, and shows the per-stage signal so you can
   verify the spec R3 timing prediction holds in a realistic
   detector pipeline.

Reference: `docs/spec_windowing_smoothing.md` § 7.4 (corrected R3
formula, Phase I.1).

## 0. Setup

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

# Make DynaSD importable from the repo root.
sys.path.append(str(Path.cwd().parent))

from DynaSD.base import DynaSDBase
from DynaSD import HFER
from tests.synthetic_data_generator import SyntheticSeizureGenerator

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

In [ ]:
def seconds_to_idx(D, w_size, w_stride):
    """Spec § 6: floor((D - w_size)/w_stride) + 1."""
    return int(np.floor((D - w_size) / w_stride)) + 1


def filter_offset_for(filter_w_idx):
    """Spec R3 parity offset for threshold-0.5 step input."""
    return 1 if filter_w_idx % 2 == 0 else 0


def predict_onset_idx(planted_idx, w_size, w_stride, filter_w, rwin_size, rwin_req):
    """Corrected R3 formula: detected onset window index for a clean
    step planted at ``planted_idx``."""
    fwi = seconds_to_idx(filter_w, w_size, w_stride)
    rsi = seconds_to_idx(rwin_size, w_size, w_stride)
    rri = seconds_to_idx(rwin_req, w_size, w_stride)
    return planted_idx + filter_offset_for(fwi) - (rsi - rri)


def annotate_onsets(ax, planted, predicted, detected, time_axis_label="window idx"):
    """Draw the three reference verticals on a panel."""
    ax.axvline(planted, color="steelblue", lw=1.2, ls=":", label=f"planted = {planted}")
    ax.axvline(predicted, color="seagreen", lw=1.2, ls="--", label=f"R3 predicted = {predicted}")
    ax.axvline(detected, color="crimson", lw=1.2, label=f"detected = {detected:g}")
    ax.set_xlabel(time_axis_label)

## 1. The full pipeline on a clean synthetic step

Plant a step from 0 → 1 at window index 50, run `get_onset_and_spread`
with default-ish parameters (`filter_w=10s, rwin_size=5s, rwin_req=4s,
threshold=0.5`), and visualize each stage of the pipeline alongside
the spec R3 prediction.

With these parameters: `filter_w_idx = 10` (even, parity offset = +1)
and spread shift `= -(rwin_size_idx - rwin_req_idx) = -1`, so the
predicted detected onset = `planted + 1 - 1 = 50`. The two shifts
cancel exactly.

In [ ]:
fs, w_size, w_stride = 1, 1.0, 1.0
n_windows = 100
planted_idx = 50
filter_w, rwin_size, rwin_req = 10.0, 5.0, 4.0
threshold = 0.5

base = DynaSDBase(fs=fs, w_size=w_size, w_stride=w_stride)

raw = np.zeros((n_windows, 1))
raw[planted_idx:, 0] = 1.0
time_idx = base.get_win_index(n_windows)
sz_prob = pd.DataFrame(raw, columns=["ch0"], index=time_idx)

onsets, sz_spread_ff = base.get_onset_and_spread(
    sz_prob, threshold=threshold,
    filter_w=filter_w, rwin_size=rwin_size, rwin_req=rwin_req,
    ret_smooth_mat=True,
)

fwi = seconds_to_idx(filter_w, w_size, w_stride)
smoothed = uniform_filter1d(raw[:, 0], size=fwi, mode="nearest")
sz_clf = (smoothed > threshold).astype(float)

detected = onsets["ch0"].iloc[0]
predicted = predict_onset_idx(planted_idx, w_size, w_stride, filter_w, rwin_size, rwin_req)

fig, axs = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
axs[0].plot(time_idx, raw[:, 0], color="black", lw=1.0)
axs[0].axhline(threshold, color="red", lw=0.6, ls="--", alpha=0.5)
axs[0].set_ylabel("raw\nsz_prob")

axs[1].plot(time_idx, smoothed, color="C0")
axs[1].axhline(threshold, color="red", lw=0.6, ls="--", alpha=0.5)
axs[1].set_ylabel(f"smoothed\n(N={fwi})")

axs[2].plot(time_idx, sz_clf, color="C2", drawstyle="steps-post")
axs[2].set_ylabel("sz_clf\n(smoothed>thr)")
axs[2].set_ylim(-0.1, 1.1)

axs[3].plot(time_idx, sz_spread_ff["ch0"].values, color="C4", drawstyle="steps-post")
axs[3].set_ylabel("sz_spread_ff\n(rwin_req of rwin_size)")
axs[3].set_ylim(-0.1, 1.1)

for ax in axs:
    ax.axvline(planted_idx, color="steelblue", lw=1.0, ls=":", alpha=0.7)
    ax.axvline(predicted, color="seagreen", lw=1.0, ls="--", alpha=0.7)
    ax.axvline(detected, color="crimson", lw=1.0, alpha=0.8)

axs[-1].set_xlabel("window index (= seconds at fs=1, w_stride=1)")
axs[0].set_title(
    f"step at idx={planted_idx}  |  filter_w={filter_w}s (idx={fwi}, parity_offset=+{filter_offset_for(fwi)})  |  "
    f"spread shift = -({seconds_to_idx(rwin_size,w_size,w_stride)}-{seconds_to_idx(rwin_req,w_size,w_stride)}) = -1  |  "
    f"R3 predicted = {predicted}, detected = {detected:g}"
)
fig.tight_layout()
plt.show()
print(f"R3 prediction matches detected: {predicted == detected}")

## 2. Filter parity (odd vs even `filter_w_idx`)

The smoothing shift at threshold 0.5 is **0 for odd** `filter_w_idx`
and **+1 for even** because the strict ``>`` comparison breaks the
half-and-half tie on the right side of the centered window. Same
step, two filter sizes — observe the smoothed crossing land at
different rows.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, filter_w_val in zip(axs, [11.0, 10.0]):  # odd then even
    raw = np.zeros((n_windows, 1))
    raw[planted_idx:, 0] = 1.0
    sz_prob = pd.DataFrame(raw, columns=["ch0"], index=base.get_win_index(n_windows))
    onsets = base.get_onset_and_spread(
        sz_prob, threshold=0.5,
        filter_w=filter_w_val, rwin_size=5.0, rwin_req=4.0,
    )
    fwi = seconds_to_idx(filter_w_val, w_size, w_stride)
    smoothed = uniform_filter1d(raw[:, 0], size=fwi, mode="nearest")
    detected = onsets["ch0"].iloc[0]
    predicted = predict_onset_idx(planted_idx, w_size, w_stride, filter_w_val, 5.0, 4.0)
    parity = "even" if fwi % 2 == 0 else "odd"
    ax.plot(time_idx, smoothed, color="C0", label="smoothed value")
    ax.axhline(0.5, color="red", lw=0.5, ls="--", alpha=0.5, label="threshold = 0.5")
    annotate_onsets(ax, planted_idx, predicted, detected)
    ax.set_xlim(40, 65)
    ax.set_title(
        f"filter_w={filter_w_val}s, idx={fwi} ({parity}), parity_offset = +{filter_offset_for(fwi)}"
    )
    ax.legend(loc="upper left", fontsize=8)
axs[0].set_ylabel("smoothed value")
fig.tight_layout()
plt.show()

## 3. Realistic post-thresholded patterns

Filter set to `filter_w = w_size` (no further smoothing), so each
panel shows the spread step's `rwin_req`-of-`rwin_size` sliding-sum
logic acting directly on the binary input. Same as
`tests/test_windowing_smoothing_spec.py::test_s74_*` cases, just
drawn out.

In [ ]:
def run_pattern(pattern, label, threshold=0.5, filter_w=1.0, rwin_size=5.0, rwin_req=4.0):
    """Run a single-channel binary pattern through the chain and return
    (sz_spread_ff array, detected onset)."""
    n = len(pattern)
    sz_prob = pd.DataFrame(
        pattern.astype(float)[:, None],
        columns=["ch0"],
        index=base.get_win_index(n),
    )
    onsets, sz_spread_ff = base.get_onset_and_spread(
        sz_prob, threshold=threshold,
        filter_w=filter_w, rwin_size=rwin_size, rwin_req=rwin_req,
        ret_smooth_mat=True,
    )
    return sz_spread_ff["ch0"].values, onsets["ch0"].iloc[0], label


# Pattern A: sparse drop-outs (1 of 5 zeros) starting at planted_idx — still detects
n = 100
k = 50
pat_sparse = np.zeros(n, dtype=int)
for i in range(k, n):
    pat_sparse[i] = 0 if (i - k) % 5 == 4 else 1

# Pattern B: dense drop-outs (alternating 1,0) — never sustains
pat_dense = np.zeros(n, dtype=int)
for i in range(k, n):
    pat_dense[i] = (i - k) % 2

# Pattern C: brief flicker at idx=30 then sustained at idx=49
pat_flicker = np.zeros(n, dtype=int)
pat_flicker[30:32] = 1
pat_flicker[49:] = 1

patterns = [
    (pat_sparse, "A. sparse drop-outs\n(1 of 5 zero)", k),
    (pat_dense, "B. dense drop-outs\n(alternating 1,0)", k),
    (pat_flicker, "C. brief flicker + sustained\n(burst at 30, real at 49)", 49),
]

fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, (pat, lab, planted) in zip(axs, patterns):
    sz_spread_ff, detected, _ = run_pattern(pat, lab)
    ax.plot(np.arange(n), pat, color="black", drawstyle="steps-post", lw=0.8, label="raw sz_clf")
    ax.plot(np.arange(n), sz_spread_ff, color="C4", drawstyle="steps-post", lw=1.5, alpha=0.8, label="sz_spread_ff")
    ax.axvline(planted, color="steelblue", lw=1.0, ls=":", label=f"planted onset = {planted}")
    if not pd.isna(detected):
        ax.axvline(detected, color="crimson", lw=1.0, label=f"detected = {detected:g}")
    else:
        ax.text(0.98, 0.85, "detected: NaN (never sustained)",
                transform=ax.transAxes, ha="right", color="crimson")
    ax.set_ylim(-0.1, 1.15)
    ax.set_ylabel(lab)
    ax.legend(loc="upper left", fontsize=8)
axs[-1].set_xlabel("window index")
fig.suptitle("Realistic post-thresholded patterns through the spread step\n"
             "filter_w=1 (no smoothing) so spread is what's being tested",
             y=1.02)
fig.tight_layout()
plt.show()

### 3b. Cascading onset across channels

Three channels with different planted onset times plus one quiet
channel. The per-channel spread step runs independently; each
detected onset matches its own planted location.

In [ ]:
n = 100
planted_per_channel = {"ch0": 20, "ch1": 35, "ch2": 50}
quiet = "ch3"
columns = list(planted_per_channel.keys()) + [quiet]
data = np.zeros((n, len(columns)))
for ch, idx in planted_per_channel.items():
    data[idx:, columns.index(ch)] = 1.0
sz_prob = pd.DataFrame(data, columns=columns, index=base.get_win_index(n))

onsets, sz_spread_ff = base.get_onset_and_spread(
    sz_prob, threshold=0.5,
    filter_w=1.0, rwin_size=5.0, rwin_req=4.0,
    ret_smooth_mat=True,
)
spread_shift = seconds_to_idx(5.0, 1.0, 1.0) - seconds_to_idx(4.0, 1.0, 1.0)

fig, ax = plt.subplots(figsize=(10, 4.5))
y_offsets = {ch: i for i, ch in enumerate(columns)}
for ch in columns:
    y = y_offsets[ch]
    ax.plot(np.arange(n), data[:, columns.index(ch)] * 0.4 + y,
            color="black", drawstyle="steps-post", lw=0.6, alpha=0.6)
    ax.plot(np.arange(n), sz_spread_ff[ch].values * 0.4 + y,
            color="C4", drawstyle="steps-post", lw=1.4, alpha=0.85)
    if ch in planted_per_channel:
        planted = planted_per_channel[ch]
        predicted = planted - spread_shift
        detected = onsets[ch].iloc[0]
        ax.scatter([planted], [y + 0.45], color="steelblue", marker="v", s=70, zorder=5)
        ax.scatter([predicted], [y + 0.45], color="seagreen", marker="v", s=70, zorder=5)
        ax.scatter([detected], [y - 0.05], color="crimson", marker="^", s=70, zorder=5)
        ax.text(planted + 1, y + 0.5,
                f"planted={planted}, predicted={predicted}, detected={detected:g}",
                fontsize=8, va="bottom")
    else:
        det = onsets[ch].iloc[0]
        ax.text(0.6, y + 0.45,
                f"detected: {'NaN' if pd.isna(det) else f'{det:g}'} (quiet channel)",
                transform=ax.get_yaxis_transform(), fontsize=8)
ax.set_yticks(list(y_offsets.values()))
ax.set_yticklabels(columns)
ax.set_xlabel("window index")
ax.set_title("Cascading planted onsets — per-channel spread step is independent")
ax.set_ylim(-0.5, len(columns) - 0.2)
fig.tight_layout()
plt.show()

## 4. Onset fidelity on a simulated polyspike seizure

Generate a 60-second synthetic iEEG signal with a window-aligned
polyspike seizure planted at `t = 30s` on the first 4 of 8 channels,
fit `HFER` on the baseline portion, run `forward(X)` then
`get_onset_and_spread`, and overlay the detected onset on top of the
planted onset for each channel.

HFER is used because the polyspike's mid-frequency content lights up
the gamma+beta vs theta+alpha ratio cleanly, separation is large,
and inference is fast on a CPU.

Threshold = 8.0 was picked offline as the midpoint of the
baseline-mean and seizure-mean of HFER on this fixture; tuning it
to the smoothed midpoint gives the cleanest spec R3 timing match.

The expected detected onset (corrected R3, with `filter_w=10s,
w_stride=0.5, rwin_size=5, rwin_req=4`):
`filter_w_idx = 19` (odd, parity offset 0), spread shift =
`-(rwin_size_idx - rwin_req_idx) * w_stride = -1.0s`. So
**predicted detected onset = 29.0s** for every planted channel.

In [ ]:
FS = 256
BASELINE_DUR = 30.0
SEIZURE_DUR = 30.0
N_CHANNELS = 8
FOCAL = [0, 1, 2, 3]
PLANTED_ONSET = BASELINE_DUR  # window-aligned

gen = SyntheticSeizureGenerator(fs=FS, random_seed=42)
combined, sz_start, sz_end = gen.generate_combined_signal(
    baseline_duration=BASELINE_DUR,
    seizure_duration=SEIZURE_DUR,
    seizure_type="polyspike",
    n_channels=N_CHANNELS,
    focal_channels=FOCAL,
    transition_duration=0.0,
)

# Workaround: SyntheticSeizureGenerator's polyspike block has a leaky
# for-loop scope (else-branch commented out); non-focal channels end
# up reusing the focal polyspike parameters. Replace non-focal
# channels with one continuous 60-second baseline so they truly stay
# baseline through both halves of the signal.
non_focal_idx = [i for i in range(N_CHANNELS) if i not in FOCAL]
if non_focal_idx:
    full_baseline = gen.generate_baseline_signal(
        n_seconds=BASELINE_DUR + SEIZURE_DUR, n_channels=len(non_focal_idx)
    )
    non_focal_cols = [combined.columns[i] for i in non_focal_idx]
    combined.loc[:, non_focal_cols] = full_baseline.values

planted_channels = [combined.columns[i] for i in FOCAL]
unplanted_channels = [c for c in combined.columns if c not in planted_channels]
print(f"signal: {combined.shape} ({BASELINE_DUR + SEIZURE_DUR}s @ {FS}Hz, {N_CHANNELS} channels)")
print(f"planted onset: {PLANTED_ONSET}s on {planted_channels}")
print(f"unplanted: {unplanted_channels}")

In [ ]:
# Plot the raw signal as a waterfall so the seizure is visible
fig, ax = plt.subplots(figsize=(11, 5))
t = np.arange(len(combined)) / FS
spacing = combined.std().median() * 8
for i, ch in enumerate(combined.columns):
    sig_norm = combined[ch].values - combined[ch].mean()
    color = "crimson" if ch in planted_channels else "black"
    ax.plot(t, sig_norm + i * spacing, color=color, lw=0.4)
ax.axvline(PLANTED_ONSET, color="steelblue", lw=1.2, ls=":", label=f"planted onset = {PLANTED_ONSET}s")
ax.set_yticks([i * spacing for i in range(N_CHANNELS)])
ax.set_yticklabels(combined.columns)
ax.set_xlabel("time (s)")
ax.set_title("Synthetic iEEG: 30s baseline → 30s polyspike seizure on Ch01-Ch04 (red); Ch05-Ch08 stay baseline")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# Run HFER
W_SIZE, W_STRIDE = 1.0, 0.5
FILTER_W, RWIN_SIZE, RWIN_REQ = 10.0, 5.0, 4.0
THRESHOLD = 8.0

model = HFER(fs=FS, w_size=W_SIZE, w_stride=W_STRIDE)
baseline_only = combined.iloc[: int(BASELINE_DUR * FS)]
model.fit(baseline_only)
sz_prob = model.forward(combined)

onsets, sz_spread_ff = model.get_onset_and_spread(
    sz_prob, threshold=THRESHOLD,
    filter_w=FILTER_W, rwin_size=RWIN_SIZE, rwin_req=RWIN_REQ,
    ret_smooth_mat=True,
)

# R3 prediction
fwi = seconds_to_idx(FILTER_W, W_SIZE, W_STRIDE)
rsi = seconds_to_idx(RWIN_SIZE, W_SIZE, W_STRIDE)
rri = seconds_to_idx(RWIN_REQ, W_SIZE, W_STRIDE)
shift_seconds = (filter_offset_for(fwi) - (rsi - rri)) * W_STRIDE
predicted_onset = PLANTED_ONSET + shift_seconds

print(f"filter_w_idx = {fwi} ({'even' if fwi % 2 == 0 else 'odd'}), "
      f"rwin_size_idx = {rsi}, rwin_req_idx = {rri}")
print(f"shift_seconds = (parity_offset {filter_offset_for(fwi)} - spread_shift {rsi - rri}) * w_stride {W_STRIDE} = {shift_seconds}s")
print(f"R3 predicted detected onset = {PLANTED_ONSET} + {shift_seconds} = {predicted_onset}s")
print()
print("Per-channel detected onsets:")
for ch in combined.columns:
    val = onsets[ch].iloc[0]
    flag = "[planted]" if ch in planted_channels else "[unplanted]"
    diff = "" if pd.isna(val) else f"  (\u0394 from R3 = {val - predicted_onset:+0.2f}s)"
    val_s = "NaN" if pd.isna(val) else f"{val:.2f}s"
    print(f"  {ch:>5}  {flag:<11}  detected = {val_s:<7}{diff}")

In [ ]:
# Visualize the detector pipeline for one planted channel
ch = planted_channels[0]  # Ch01
raw = sz_prob[ch].values
smoothed = uniform_filter1d(raw, size=fwi, mode="nearest")
sz_clf_binary = (smoothed > THRESHOLD).astype(float)
spread_ch = sz_spread_ff[ch].values
t = sz_prob.index.values  # seconds
detected_ch = onsets[ch].iloc[0]

fig, axs = plt.subplots(4, 1, figsize=(11, 8.5), sharex=True)
axs[0].plot(t, raw, color="black", lw=0.6, label=f"HFER raw on {ch}")
axs[0].axhline(THRESHOLD, color="red", lw=0.6, ls="--", alpha=0.5, label=f"threshold = {THRESHOLD}")
axs[0].set_yscale("log")
axs[0].set_ylabel("raw HFER\n(log scale)")
axs[0].legend(loc="upper left", fontsize=8)

axs[1].plot(t, smoothed, color="C0", label=f"smoothed (uniform_filter1d, N={fwi})")
axs[1].axhline(THRESHOLD, color="red", lw=0.6, ls="--", alpha=0.5)
axs[1].set_yscale("log")
axs[1].set_ylabel("smoothed\n(log scale)")
axs[1].legend(loc="upper left", fontsize=8)

axs[2].plot(t, sz_clf_binary, color="C2", drawstyle="steps-post")
axs[2].set_ylabel("sz_clf\n(smoothed > thr)")
axs[2].set_ylim(-0.1, 1.1)

axs[3].plot(t, spread_ch, color="C4", drawstyle="steps-post")
axs[3].set_ylabel("sz_spread_ff")
axs[3].set_ylim(-0.1, 1.1)

for ax in axs:
    ax.axvline(PLANTED_ONSET, color="steelblue", lw=1.2, ls=":", alpha=0.7)
    ax.axvline(predicted_onset, color="seagreen", lw=1.2, ls="--", alpha=0.7)
    if not pd.isna(detected_ch):
        ax.axvline(detected_ch, color="crimson", lw=1.2, alpha=0.85)
axs[-1].set_xlabel("time (s)")
title = (
    f"HFER pipeline on {ch} (planted): planted = {PLANTED_ONSET}s, "
    f"R3 predicted = {predicted_onset}s, detected = "
    f"{'NaN' if pd.isna(detected_ch) else f'{detected_ch:.2f}s'}"
)
axs[0].set_title(title)
fig.tight_layout()
plt.show()

In [ ]:
# Per-channel onset summary as a horizontal lollipop chart
channels_sorted = list(combined.columns)
ys = np.arange(len(channels_sorted))
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.axvline(PLANTED_ONSET, color="steelblue", lw=1.2, ls=":", label=f"planted = {PLANTED_ONSET}s")
ax.axvline(predicted_onset, color="seagreen", lw=1.2, ls="--", label=f"R3 predicted = {predicted_onset}s")
for y, ch in zip(ys, channels_sorted):
    val = onsets[ch].iloc[0]
    is_planted = ch in planted_channels
    if pd.isna(val):
        ax.text(predicted_onset + 1, y, "NaN (no detection)",
                color="gray" if not is_planted else "crimson",
                fontsize=9, va="center")
    else:
        color = "crimson" if is_planted else "darkorange"
        ax.scatter([val], [y], color=color, s=80, zorder=5)
        ax.plot([predicted_onset, val], [y, y], color=color, lw=1.2, alpha=0.5)
        ax.text(val + 0.3, y, f"{val:.2f}s", fontsize=9, va="center", color=color)
ax.set_yticks(ys)
ax.set_yticklabels([f"{ch} {'(planted)' if ch in planted_channels else '(unplanted)'}" for ch in channels_sorted])
ax.set_xlabel("time (s)")
ax.set_xlim(predicted_onset - 5, max([onsets[ch].iloc[0] for ch in channels_sorted if not pd.isna(onsets[ch].iloc[0])] + [predicted_onset + 2]) + 2)
ax.set_title("Per-channel detected onset vs. R3 prediction (planted = 30.0s)")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
plt.show()

## What this notebook demonstrates

- **Section 1** shows that for a clean step input at threshold 0.5,
  the smoothing/threshold/spread chain in `get_onset_and_spread`
  produces the detected onset exactly where the corrected R3
  formula predicts. With even `filter_w_idx`, the +1 parity
  smoothing offset cancels the -1 spread shift; the detected onset
  equals the planted step row exactly.
- **Section 2** shows the parity rule: odd `filter_w_idx` produces
  no smoothing shift at threshold 0.5, even `filter_w_idx` shifts
  the smoothed crossing by +1 row.
- **Section 3** shows the spread step's `rwin_req` filter at work:
  sparse drop-outs that still satisfy `rwin_req` detect normally,
  dense drop-outs and brief flickers don't, and per-channel
  cascades work independently.
- **Section 4** runs the same chain on a real detector
  (`HFER.forward(X)`) on a synthetic polyspike seizure. Detected
  onset on planted channels lands within ~1 stride of the R3
  prediction; unplanted channels return NaN. Any deviation from
  the R3 prediction here is a function of feature-space noise
  (the smoothed crossing point of the noisy HFER signal isn't
  exactly at the smoothed-midpoint), not the smoothing/spread
  machinery itself.